In [ ]:
import pyvista as pv

pv.set_jupyter_backend("static")

%load_ext autoreload
%autoreload 2

## Process-structure-property demo

Run a process-structure-property simulation using `RosenthalSolidificationModel` and `SmallStrainFFT` with an `Elastic` constitutive model.

In [ ]:
from materialite import Material, Vector, add_ipf_colors_field
from materialite.models import RosenthalSolidificationModel
from materialite.models.laser_path import LaserPath
from materialite import Order4SymmetricTensor
from materialite.models.small_strain_fft import SmallStrainFFT, Elastic, LoadSchedule
import numpy as np

Create a `Material`

In [ ]:
material = Material(
    spacing=[5.0e-6, 5.0e-6, 5.0e-6], dimensions=[40, 90, 33]
).create_random_integer_field("grain", 1, 10000)

Create the process-structure model, which includes two layers of an additive manufacturing build. See the Rosenthal Solidification Demo and Laser Path Demo for more details.

In [ ]:
hatch_spacing = 110.0e-6
time_between_scans = 1.0e-5
path_inputs = {
    "x_bounds": [material.origin[0], material.sizes[0]],
    "y_bounds": [material.origin[1], material.sizes[1]],
    "hatch_spacing": hatch_spacing,
    "velocity": 0.96,
    "power": 285.0,
    "end_offset": 900.0e-6,
    "domain_offset": hatch_spacing / 2,
    "delay_between_scans": time_between_scans,
}
path1 = LaserPath.angled_raster_scan(angle=67.0, z_height=120.0e-6, **path_inputs)
path2 = LaserPath.angled_raster_scan(
    angle=134.0,
    z_height=160.0e-6,
    start_time=path1.total_time + time_between_scans,
    **path_inputs
)
laser_path = LaserPath.from_list([path1, path2])
time_step_duration = 5.2e-6
output_times = (
    laser_path.end_times - laser_path.start_times
) / 2 + laser_path.start_times
thermophysical_properties = {
    "conductivity": 29.6,
    "diffusivity": 4.79e-6,
    "absorptivity": 0.46,
    "melt_temperature": 1609.0,
}
solidification_properties = {"prefactor": 1.2e-5, "exponent": 2.6}
initial_temperature = 300.0
process_model = RosenthalSolidificationModel(
    laser_path=laser_path,
    initial_temperature=300.0,
    thermophysical_properties=thermophysical_properties,
    solidification_properties=solidification_properties,
    time_step_duration=time_step_duration,
    output_times=output_times,
)

Create the structure-property model. See the Constitutive Model demos for more details.

In [ ]:
stiffness = Order4SymmetricTensor.from_cubic_constants(C11=243.3, C12=156.7, C44=117.8)
elastic_model = Elastic(stiffness)
load_schedule = LoadSchedule.from_constant_uniaxial_strain_rate(
    magnitude=1.0, direction="y"
)
mechanical_model = SmallStrainFFT(
    load_schedule=load_schedule, end_time=0.001, constitutive_model=elastic_model
)

Put everything together. Since the process-structure model does not include crystallographic texture, we assign random orientations to each grain at the end of the model and add a field with the IPF colors with respect to the loading direction. We also crop out the bottom of the model, which contains unmelted voxels from below the first layer in the process-structure model.

In [ ]:
material = (
    material.run(process_model)
    .assign_random_orientations(region_label="grain")
    .run(add_ipf_colors_field, direction=Vector([0, 1, 0]))
    .crop(z=[70.0e-6, np.inf])
    .run(
        mechanical_model,
        linear_solver_tolerance=1.0e-8,
        global_tolerance=1.0,
        strain_correction_tolerance=1.0,
    )
)

Plot the stress in the loading direction.

In [ ]:
material.plot("stress", component=1)

Plot the IPF map

In [ ]:
material.plot("ipf_color", kind="ipf_map")